In [2]:
import pandas as pd
import numpy as np
import json, math, textwrap
from math import sqrt
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs


import sys
from pathlib import Path


ROOT = Path.cwd().resolve()
while not (ROOT / "paths.py").exists():
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))
from paths import OUTPUT_DIR

# ============================================================
# INSTELLINGEN
# ============================================================


CSV_PATH = OUTPUT_DIR/"model_coefficients.csv"   # <- alleen dit aanpassen per jaar
OUTPUT_HTML = "sem_dashboard.html"

# ============================================================
# HELPERS
# ============================================================
def safe_float(x):
    if x is None: 
        return None
    s = str(x).strip()
    if s in ["", "-", "NA", "NaN", "nan", "None", "null"]:
        return None
    try:
        return float(s)
    except:
        return None

def sanitize(obj):
    if isinstance(obj, dict): 
        return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list): 
        return [sanitize(v) for v in obj]
    if isinstance(obj, tuple): 
        return [sanitize(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj): 
            return None
        return obj
    return obj

def norm_colnames(df): 
    return {c.lower(): c for c in df.columns}

def req_col(df, name):
    cols = norm_colnames(df)
    if name.lower() not in cols:
        raise ValueError(f"Kolom '{name}' ontbreekt. Gevonden: {list(df.columns)}")
    return cols[name.lower()]

def wrap_html(text, width=14, max_lines=3):
    t = str(text).replace("_", " ").strip()
    lines = textwrap.wrap(t, width=width)[:max_lines]
    return "<br>".join(lines) if lines else t

def wrap_indicator(text, width=14, max_lines=2):
    t = str(text).replace("_", " ").strip()
    lines = textwrap.wrap(t, width=width)[:max_lines]
    return "<br>".join(lines) if lines else t

def p_to_stars(p):
    p = safe_float(p)
    if p is None: 
        return ""
    if p < 0.001: 
        return "***"
    if p < 0.01:  
        return "**"
    if p < 0.05:  
        return "*"
    return ""

def clamp(a, lo, hi): 
    return max(lo, min(hi, a))

def edge_width(beta):
    b = abs(beta)
    return 1.2 + 2.4 * clamp(b, 0.0, 0.9)

def dist(a,b):
    return sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def unit_normal(dx,dy):
    L = sqrt(dx*dx+dy*dy) if (dx*dx+dy*dy) else 1.0
    return (-dy/L, dx/L)

def label_box_size(label: str, base=32, per_char=2.4, min_size=32, max_size=52):
    """
    Marker size (witte label-vakjes) per padlabel, op basis van tekstlengte.
    """
    s = str(label)
    size = base + per_char * max(0, len(s) - 4)  # -4 = '±0.00' baseline
    return int(max(min_size, min(max_size, size)))

# ============================================================
# READ CSV
# ============================================================
df = pd.read_csv(CSV_PATH)

c_lval = req_col(df, "lval")
c_op   = req_col(df, "op")
c_rval = req_col(df, "rval")
c_est  = req_col(df, "Estimate")
cols = norm_colnames(df)
c_p  = cols.get("p-value", None)

keep = [c_lval, c_op, c_rval, c_est] + ([c_p] if c_p else [])
work = df[keep].copy()
work.columns = ["lval","op","rval","estimate"] + (["p"] if c_p else [])

work["estimate"] = work["estimate"].apply(safe_float)
if "p" in work.columns:
    work["p"] = work["p"].apply(safe_float)

reg = work[work["op"]=="~"].copy()
var = work[work["op"]=="~~"].copy()

# ============================================================
# AUTO DETECT LATENTS
# ============================================================
predictors = set(reg["rval"].astype(str))
outcomes   = set(reg["lval"].astype(str))
variances  = set(var.loc[var["lval"].astype(str)==var["rval"].astype(str), "lval"].astype(str))

latents = sorted([x for x in predictors if (x in outcomes) or (x in variances)])
if len(latents)==0:
    latents = list(reg["rval"].value_counts().head(6).index.astype(str))

latent_set = set(latents)

LABELS = {
    "Stressors": "Stressoren",
    "Energy_Sources": "Energiebronnen",
    "Response_to_Stress": "Stressreacties",
    "Wellbeing": "Welbevinden",
    "Negative_Outcomes": "Negatieve uitkomsten",
    "Positive_Outcomes": "Positieve uitkomsten",
}
def latent_title(x):
    return LABELS.get(x, x.replace("_"," "))

# ============================================================
# SPLIT
# ============================================================
paths = reg[(reg["lval"].astype(str).isin(latent_set)) & (reg["rval"].astype(str).isin(latent_set))].copy()
paths = paths[paths["estimate"].notna()]

loadings = reg[(reg["rval"].astype(str).isin(latent_set)) & (~reg["lval"].astype(str).isin(latent_set))].copy()
loadings = loadings[loadings["estimate"].notna()]

resid = var[(var["lval"].astype(str)==var["rval"].astype(str)) & (~var["lval"].astype(str).isin(latent_set))].copy()
resid_map = dict(zip(resid["lval"].astype(str), resid["estimate"].apply(safe_float)))

# ============================================================
# POSITIONS (JD-R layout)
# ============================================================
preferred_pos = {
    "Stressors": (-9.0,  3.6),
    "Response_to_Stress": (0.0,  3.6),
    "Negative_Outcomes": ( 9.0,  3.6),
    "Energy_Sources": (-9.0, -3.6),
    "Wellbeing": (0.0, -3.6),
    "Positive_Outcomes": ( 9.0, -3.6),
}
pos={}
unknown=[]
for l in latents:
    if l in preferred_pos: 
        pos[l]=preferred_pos[l]
    else: 
        unknown.append(l)

if unknown:
    cx,cy = 14.0,0.0
    R=4.2
    for i,l in enumerate(unknown):
        ang = 2*np.pi*i/max(1,len(unknown))
        pos[l]=(cx+R*np.cos(ang), cy+R*np.sin(ang))

node_centers = {l: pos[l] for l in latents}

# ============================================================
# LABEL PLACEMENT
# ============================================================
NODE_RADIUS = 1.65
CLEARANCE   = 0.85
LABEL_OFF   = 0.95

def push_out(pt):
    x,y = pt
    for _,c in node_centers.items():
        d = dist((x,y), c)
        min_d = NODE_RADIUS + CLEARANCE
        if d < min_d:
            vx,vy = x-c[0], y-c[1]
            vL = sqrt(vx*vx+vy*vy) if (vx*vx+vy*vy) else 1.0
            push = (min_d - d) + 0.25
            x += (vx/vL)*push
            y += (vy/vL)*push
    return (x,y)

def label_for_edge(p0, p2, idx):
    dx,dy = p2[0]-p0[0], p2[1]-p0[1]
    nx,ny = unit_normal(dx,dy)

    is_horizontal = abs(dy) < 0.25 and abs(dx) > 2.0

    if is_horizontal:
        t = 0.50
        sign = 1 if p0[1] >= 0 else -1
        x = p0[0] + dx*t
        y = p0[1] + dy*t
        lx = x + nx*LABEL_OFF*sign
        ly = y + ny*LABEL_OFF*sign
    else:
        t = 0.35
        x = p0[0] + dx*t
        y = p0[1] + dy*t

        slope_pos = (dx*dy) > 0
        base_sign = 1 if p0[1] >= 0 else -1
        sign = base_sign if slope_pos else -base_sign

        stagger = (idx % 3 - 1) * 0.18
        lx = x + nx*(LABEL_OFF + stagger)*sign
        ly = y + ny*(LABEL_OFF + stagger)*sign

    return push_out((lx,ly))

# ============================================================
# OVERVIEW FIG
# ============================================================
edge_traces=[]
label_x=[]; label_y=[]; label_text=[]; label_hover=[]

for idx, r in paths.reset_index(drop=True).iterrows():
    s = str(r["rval"]); t = str(r["lval"])
    beta = safe_float(r["estimate"])
    pval = safe_float(r["p"]) if "p" in paths.columns else None
    if beta is None or s not in pos or t not in pos:
        continue

    p0=pos[s]; p2=pos[t]

    edge_traces.append(go.Scatter(
        x=[p0[0], p2[0]], y=[p0[1], p2[1]],
        mode="lines",
        line=dict(width=edge_width(beta), color="rgba(0,0,0,0.55)"),
        hoverinfo="text",
        hovertext="<b>Regression path</b><br>{} → {}<br>β = {:+.3f}{}{}".format(
            latent_title(s), latent_title(t), beta, p_to_stars(pval),
            ("<br>p = {:.3g}".format(pval) if pval is not None else "")
        ),
        showlegend=False
    ))

    lx,ly = label_for_edge(p0,p2,idx)
    label_x.append(lx); label_y.append(ly)
    label_text.append("{:+.2f}{}".format(beta, p_to_stars(pval)))
    label_hover.append("<b>Regression path</b><br>{} → {}<br>β = {:+.3f}{}{}".format(
        latent_title(s), latent_title(t), beta, p_to_stars(pval),
        ("<br>p = {:.3g}".format(pval) if pval is not None else "")
    ))

label_sizes = [label_box_size(t) for t in label_text]

labels_scatter = go.Scatter(
    x=label_x, y=label_y,
    mode="markers+text",
    text=label_text,
    textposition="middle center",
    textfont=dict(size=10.6, color="#111"),
    marker=dict(
        size=label_sizes,
        symbol="square",
        color="rgba(255,255,255,0.98)",
        line=dict(width=1.2, color="rgba(0,0,0,0.20)")
    ),
    hoverinfo="text",
    hovertext=label_hover,
    showlegend=False
)

node_trace = go.Scatter(
    x=[pos[l][0] for l in latents],
    y=[pos[l][1] for l in latents],
    mode="markers+text",
    text=[wrap_html(latent_title(l), width=14, max_lines=3) for l in latents],
    textposition="middle center",
    textfont=dict(size=13, color="white"),
    marker=dict(size=160, color="#234a6f", line=dict(width=2, color="#0b0c10")),
    hoverinfo="text",
    hovertext=[f"<b>{latent_title(l)}</b><br>Latent variable: {l}" for l in latents],
    showlegend=False
)

overview_fig = go.Figure(data=edge_traces + [labels_scatter, node_trace])
overview_fig.update_layout(
    title="SEM Dashboard – Overzicht",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    height=760,
    margin=dict(l=20, r=20, t=70, b=20),
    plot_bgcolor="rgba(248,250,255,1)",
    paper_bgcolor="rgba(248,250,255,1)",
)
overview_node_trace_index = len(overview_fig.data) - 1

# ============================================================
# DETAIL FIGS
# ============================================================
detail_figs={}
detail_meta={}
for latent in latents:
    items = loadings[loadings["rval"].astype(str)==latent].copy().sort_values("estimate", ascending=False)

    ind_list=[]
    for _, rr in items.iterrows():
        ind=str(rr["lval"])
        ld=safe_float(rr["estimate"])
        pv=safe_float(rr["p"]) if "p" in items.columns else None
        if ld is None: 
            continue
        ind_list.append((ind, ld, pv, safe_float(resid_map.get(ind))))

    n=len(ind_list)
    top_n=int(np.ceil(n/2))
    top=ind_list[:top_n]
    bot=ind_list[top_n:]

    spacing_x=3.35
    top_y=3.7
    bot_y=-3.7
    cx,cy=0.0,0.0

    def place_row(row_items, y):
        nodes=[]
        for j,(name,ld,pv,rv) in enumerate(row_items):
            x=(j-(len(row_items)-1)/2)*spacing_x
            nodes.append({"name":name,"x":x,"y":y,"loading":ld,"p":pv,"resid":rv})
        return nodes

    ind_nodes = place_row(top, top_y) + place_row(bot, bot_y)

    ex,ey=[],[]
    for n1 in ind_nodes:
        ex += [cx, n1["x"], None]
        ey += [cy, n1["y"], None]

    edges = go.Scatter(x=ex, y=ey, mode="lines",
                       line=dict(width=1.25, color="rgba(0,0,0,0.50)"),
                       hoverinfo="none", showlegend=False)

    center = go.Scatter(
        x=[cx], y=[cy], mode="markers+text",
        text=[wrap_html(latent_title(latent), width=14, max_lines=3)],
        textposition="middle center",
        textfont=dict(size=14, color="white"),
        marker=dict(size=170, color="#234a6f", line=dict(width=2, color="#0b0c10")),
        hoverinfo="text",
        hovertext=[f"<b>{latent_title(latent)}</b><br>Latent variable: {latent}"],
        showlegend=False
    )

    ind_text=[wrap_indicator(n1["name"], width=14, max_lines=2) for n1 in ind_nodes]
    ind_hover=[]
    for n1 in ind_nodes:
        ind_hover.append(
            "<b>{}</b><br>Loading (λ): {:+.3f}{}{}".format(
                n1["name"], n1["loading"], p_to_stars(n1["p"]),
                ("<br>p = {:.3g}".format(n1["p"]) if n1["p"] is not None else "")
            ) + (("<br>Residual variance: {:.3f}".format(n1["resid"])) if n1["resid"] is not None else "")
        )

    inds = go.Scatter(
        x=[n1["x"] for n1 in ind_nodes],
        y=[n1["y"] for n1 in ind_nodes],
        mode="markers+text",
        text=ind_text,
        textposition="middle center",
        textfont=dict(size=10.4, color="#111"),
        marker=dict(size=84, symbol="square",
                    color="#f7e7a0", line=dict(width=1.5, color="#c29c00")),
        hoverinfo="text",
        hovertext=ind_hover,
        showlegend=False
    )

    highlight = go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=98, symbol="square-open",
                    color="rgba(0,0,0,0)",
                    line=dict(width=4, color="rgba(17,24,39,0.95)")),
        hoverinfo="none",
        showlegend=False
    )

    fig = go.Figure(data=[edges, center, inds, highlight])
    fig.update_layout(
        title=f"Detail – {latent_title(latent)}",
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        height=760,
        margin=dict(l=20, r=20, t=70, b=20),
        plot_bgcolor="rgba(248,250,255,1)",
        paper_bgcolor="rgba(248,250,255,1)",
    )

    detail_figs[latent]=fig
    detail_meta[latent]={
        "indicatorIndexByName": {n1["name"].lower(): i for i,n1 in enumerate(ind_nodes)},
        "indicatorTraceIndex": 2,
        "highlightTraceIndex": 3
    }

bundle = {
    "overviewFig": sanitize(overview_fig.to_dict()),
    "detailFigs": {k: sanitize(v.to_dict()) for k,v in detail_figs.items()},
    "latents": latents,
    "latentTitles": {l: latent_title(l) for l in latents},
    "overviewNodeTraceIndex": overview_node_trace_index,
    "detailMeta": detail_meta
}
bundle_json = json.dumps(bundle)
plotly_js = get_plotlyjs()

# ============================================================
# HTML
# ============================================================
HTML = r"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>SEM Dashboard</title>
<script>__PLOTLY_JS__</script>
<style>
  :root { --bg:#f4f7ff; --card:#fff; --text:#111827; --muted:#6b7280; --stroke:rgba(0,0,0,0.10);
          --shadow:0 14px 38px rgba(0,0,0,0.10); --btnbg:rgba(255,255,255,0.92); --btnbd:rgba(0,0,0,0.14); }
  body.dark { --bg:#0f1220; --card:#171a2b; --text:#e9ecff; --muted:#b9bed6; --stroke:rgba(255,255,255,0.10);
              --shadow:0 18px 44px rgba(0,0,0,0.55); --btnbg:rgba(23,26,43,0.92); --btnbd:rgba(255,255,255,0.12); }
  body { margin:0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; background:var(--bg); color:var(--text); }
  .wrap { max-width:1400px; margin:18px auto; padding:12px 18px 26px 18px; }
  .top { display:flex; justify-content:space-between; align-items:center; gap:12px; margin-bottom:12px; }
  .h1 { font-size:22px; font-weight:760; }
  .sub { font-size:13px; color:var(--muted); }
  .actions { display:flex; gap:8px; flex-wrap:wrap; align-items:center; }
  .btn { border-radius:999px; border:1px solid var(--btnbd); padding:8px 12px; background:var(--btnbg); cursor:pointer; font-size:13px; user-select:none; }
  .grid { display:grid; grid-template-columns:minmax(0,2.4fr) minmax(420px,1fr); gap:14px; align-items:start; }
  .card { background:var(--card); border-radius:18px; box-shadow:var(--shadow); border:1px solid var(--stroke); padding:12px 14px; }
  #plot { height:760px; }
  .controlrow { display:flex; gap:10px; flex-wrap:wrap; align-items:center; margin:10px 0 6px 0; }
  .chip { border:1px solid var(--stroke); border-radius:999px; padding:6px 10px; font-size:12.5px; color:var(--muted);
          background: rgba(255,255,255,0.35); }
  body.dark .chip { background: rgba(255,255,255,0.06); }
  .input { flex:1; min-width:240px; border:1px solid var(--stroke); border-radius:12px; padding:10px 12px;
           background: rgba(255,255,255,0.75); outline:none; color:var(--text); }
  body.dark .input { background: rgba(255,255,255,0.06); }
  .panel h3 { margin:6px 0 8px 0; font-size:16px; }
  .panel p, .panel li { font-size:12.6px; color:var(--muted); line-height:1.6; }
  .panel ul { margin:8px 0 0 0; padding-left:18px; }
  .hr { height:1px; background:var(--stroke); margin:12px 0; }
</style>
</head>
<body>
<div class="wrap">
  <div class="top">
    <div>
      <div class="h1">SEM Dashboard</div>
    </div>
    <div class="actions">
      <div class="btn" id="backBtn" style="display:none;">← Terug</div>
      <div class="btn" id="resetBtn">Reset zoom</div>
      <div class="btn" id="themeBtn">🌗 Dark</div>
    </div>
  </div>

  <div class="grid">
    <div class="card">
      <div class="controlrow">
        <input class="input" id="searchBox" placeholder="Zoek: latent (overzicht) of indicator (detail) en druk Enter" />
        <span class="chip" id="crumb">Overzicht</span>
      </div>
      <div id="plot"></div>
    </div>

    <div class="card panel">
      <h3 id="panelTitle">Uitleg (SEM)</h3>
      <div id="panelBody">
        <p>
          Een <b>latente variabele</b> is een onderliggend concept dat je niet direct observeert (zoals stress),
          maar afleidt uit meerdere <b>indicatoren</b> (items/vragen). In SEM schat je zowel (1) hoe goed indicatoren
          het concept meten (<b>factor loadings λ</b>) als (2) hoe concepten elkaar beïnvloeden (<b>regressiepaden β</b>).
        </p>
        <div class="hr"></div>
        <ul>
          <li><b>β (op paden)</b>: gestandaardiseerde effectgrootte tussen latente variabelen.</li>
          <li><b>Significantie</b>: * p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001.</li>
          <li>Open een bol voor details over λ en residual variance.</li>
        </ul>
      </div>
    </div>
  </div>
</div>

<script>
  const BUNDLE = __BUNDLE_JSON__;

  const plotDiv = document.getElementById('plot');
  const backBtn = document.getElementById('backBtn');
  const resetBtn= document.getElementById('resetBtn');
  const themeBtn= document.getElementById('themeBtn');
  const searchBox = document.getElementById('searchBox');
  const crumb = document.getElementById('crumb');

  let view = "overview";
  let currentLatent = null;

  function bindOverviewEvents() {
    if (typeof plotDiv.on !== "function") return;
    if (typeof plotDiv.removeAllListeners === "function") plotDiv.removeAllListeners('plotly_click');
    plotDiv.on('plotly_click', function(evt) {
      const pt = evt && evt.points ? evt.points[0] : null;
      if (!pt) return;
      if (pt.curveNumber === BUNDLE.overviewNodeTraceIndex) {
        const key = BUNDLE.latents[pt.pointIndex];
        if (key) renderDetail(key);
      }
    });
  }

  function renderOverview() {
    view = "overview";
    currentLatent = null;
    crumb.textContent = "Overzicht";
    backBtn.style.display = "none";
    Plotly.newPlot(plotDiv, BUNDLE.overviewFig.data, BUNDLE.overviewFig.layout, {displayModeBar:true, responsive:true})
      .then(bindOverviewEvents);
  }

  function renderDetail(latentKey) {
    view = "detail";
    currentLatent = latentKey;
    crumb.textContent = "Detail: " + (BUNDLE.latentTitles[latentKey] || latentKey);
    backBtn.style.display = "inline-block";
    Plotly.newPlot(plotDiv, BUNDLE.detailFigs[latentKey].data, BUNDLE.detailFigs[latentKey].layout, {displayModeBar:true, responsive:true});
  }

  function resetZoom() {
    Plotly.relayout(plotDiv, {"xaxis.autorange": true, "yaxis.autorange": true});
  }

  function highlightIndicator(idx) {
    const meta = BUNDLE.detailMeta[currentLatent];
    const tInd = meta.indicatorTraceIndex;
    const tHi  = meta.highlightTraceIndex;
    const fig = BUNDLE.detailFigs[currentLatent];
    const x = fig.data[tInd].x[idx];
    const y = fig.data[tInd].y[idx];
    Plotly.restyle(plotDiv, {'x': [[x]], 'y': [[y]]}, [tHi]);
    Plotly.relayout(plotDiv, {"xaxis.range": [x-4.2, x+4.2], "yaxis.range": [y-3.6, y+3.6]});
  }

  function doSearch(q) {
    const query = (q || "").trim().toLowerCase();
    if (!query) return;

    if (view === "overview") {
      for (let i=0; i<BUNDLE.latents.length; i++) {
        const key = BUNDLE.latents[i];
        const title = (BUNDLE.latentTitles[key] || key).toLowerCase();
        if (title.includes(query) || key.toLowerCase().includes(query)) { renderDetail(key); return; }
      }
      return;
    }

    if (view === "detail" && currentLatent) {
      const meta = BUNDLE.detailMeta[currentLatent];
      let idx = meta.indicatorIndexByName[query];
      if (idx == null) {
        for (const k in meta.indicatorIndexByName) {
          if (k.includes(query)) { idx = meta.indicatorIndexByName[k]; break; }
        }
        if (idx == null) return;
      }
      highlightIndicator(idx);
    }
  }

  backBtn.addEventListener('click', renderOverview);
  resetBtn.addEventListener('click', resetZoom);
  themeBtn.addEventListener('click', function() {
    document.body.classList.toggle('dark');
    themeBtn.textContent = document.body.classList.contains('dark') ? "🌞 Light" : "🌗 Dark";
  });
  searchBox.addEventListener('keydown', function(e) { if (e.key === "Enter") doSearch(searchBox.value); });

  window.addEventListener("load", function() { renderOverview(); });
</script>
</body>
</html>
"""

html = HTML.replace("__PLOTLY_JS__", plotly_js).replace("__BUNDLE_JSON__", bundle_json)
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print("✅ Klaar:", OUTPUT_HTML)
print("Open het bestand in je browser (dubbelklik).")


✅ Klaar: sem_dashboard.html
Open het bestand in je browser (dubbelklik).
